<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/banners/banner_customer_churn_nb1_solution.png" width="100%"/>
</div>

# Notebook 1 — SQL Analytics : Customer Churn Analytics — *Solution*

## Contexte métier

**IvoirCom** est un opérateur télécom mobile fictif basé à Abidjan, opérant dans 5 villes ivoiriennes (Abidjan, Bouaké, Yamoussoukro, San-Pédro, Korhogo). Sa direction commerciale alerte : **12 % de la base abonnés churn chaque trimestre**, soit un coût de rétention/acquisition (CAC perdu) estimé à 85 000 FCFA par départ.

Le formateur joue le rôle d'analyste data fraîchement embauché. Mission :

1. **Comprendre** la composition de la base (volumétrie, segments).
2. **Mesurer** le taux de churn par offre, ville, tranche d'âge.
3. **Identifier** les signaux avant-coureurs (consommation en baisse, réclamations non résolues).
4. **Segmenter** les abonnés via RFM (Récence facturation, Fréquence réclamations, Monétaire ARPU).
5. **Recommander** 3 leviers d'action chiffrés à la direction commerciale.

## Données disponibles

| Table | Lignes | Description |
|---|---|---|
| `clients` | ~8 030 | Abonnés (id, ville, offre, date souscription, statut) — contient 30 doublons et 5 âges négatifs |
| `offres` | 6 | Catalogue offres (Pulse, Connect, Premium, Pro, Étudiant, Senior) |
| `factures` | 138 284 | Facturation mensuelle 24 mois (2 % de montants nuls) |
| `consommation_mensuelle` | 137 044 | Voix / SMS / Data par client par mois (8 % de clients ont des trous) |
| `reclamations` | 9 791 | Tickets support (3 % de délais de résolution négatifs) |

## Outils

- **DuckDB** + **JupySQL** (`%%sql` magic) — analytique SQL en mémoire, lecture CSV native.
- **pandas** — manipulation tabulaire complémentaire et visualisations.
- **matplotlib / seaborn** — graphiques.

Plan du notebook (7 sections) : Exploration & Nettoyage → KPIs globaux → Segmentation → Cohortes → Signaux Réclamations → RFM Telecom → Synthèse.

---
## 0. Setup — imports, paramétrage, connexion DuckDB

### 🔧 MÉTHODE — pourquoi DuckDB plutôt que pandas seul ?

DuckDB exécute du SQL pur sur des CSV directement, sans charger d'ETL préalable. Il gère les fenêtres analytiques (`RANK`, `NTILE`, `LAG`), les `CTE` chaînées, et les agrégations multi-grain bien plus lisibles qu'en pandas. JupySQL relie DuckDB aux cellules `%%sql` du notebook tout en exposant les résultats comme des `DataFrame` pandas pour la visualisation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

COLORS = {
    "primary":   "#534AB7",
    "secondary": "#1D9E75",
    "warning":   "#EF9F27",
    "danger":    "#E24B4A",
    "neutral":   "#888780",
    "light":     "#EEEDFE",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F9F9F8",
    "axes.grid":        True,
    "grid.alpha":       0.35,
    "font.size":        11,
})  

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/logitrack_analytics/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅') 

In [ ]:
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%sql duckdb:///

### 🔧 MÉTHODE — chargement des CSV en tables DuckDB

`read_csv_auto` détecte automatiquement les types et les en-têtes. On force `parse_dates` côté pandas via une vue typée DuckDB pour les colonnes date — sinon DuckDB les laisse en `VARCHAR` et `DATEDIFF` plante. La syntaxe `CAST(col AS DATE)` est appliquée dans la vue.

In [ ]:
BASE_URL   = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/data/'


conn = duckdb.connect()
conn.execute(f"""
    CREATE TABLE clients_raw AS SELECT * FROM read_csv_auto('{BASE_URL}clients.csv');
    CREATE TABLE offres     AS SELECT * FROM read_csv_auto('{BASE_URL}offres.csv');
    CREATE TABLE factures_raw        AS SELECT * FROM read_csv_auto('{BASE_URL}factures.csv');
    CREATE TABLE consommation AS SELECT * FROM read_csv_auto('{BASE_URL}consommation_mensuelle.csv');
    CREATE TABLE reclamations_raw AS SELECT * FROM read_csv_auto('{BASE_URL}reclamations.csv');
""")

n = conn.execute('SELECT COUNT(*) FROM consommation').fetchone()[0]
print(f'✅ {n:,} livraisons chargées dans DuckDB')

%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')

In [ ]:
%%sql

SELECT 'clients' AS table_name, COUNT(*) AS n FROM clients_raw
UNION ALL SELECT 'offres', COUNT(*) FROM offres
UNION ALL SELECT 'factures', COUNT(*) FROM factures_raw
UNION ALL SELECT 'consommation', COUNT(*) FROM consommation
UNION ALL SELECT 'reclamations', COUNT(*) FROM reclamations_raw;

> 💡 **INTERPRÉTATION** *(à finaliser après exécution — chiffres attendus : clients ~8 030, factures 138 284, consommation 137 044, reclamations 9 791, offres 6)*

> 🏥 **MÉTIER** — La volumétrie correspond à 24 mois d'activité opérationnelle d'un opérateur de taille moyenne. Le ratio `factures / clients ≈ 17 mois moyens` est cohérent avec un mix de clients récents et d'historiques sur la période.

---
## 1. Exploration & nettoyage des anomalies

Avant tout calcul de KPI, on inventorie et corrige les défauts qualité connus. Une donnée non nettoyée fausse silencieusement les indicateurs (un âge négatif décale les segments, un montant nul tire l'ARPU vers le bas, un délai négatif rend la médiane absurde).

### 1.1 Doublons clients (suffixe `_DUP`) et âges négatifs

#### 🔧 MÉTHODE

Le générateur a inséré 30 doublons (id_client se terminant par `_DUP`) et 5 âges négatifs (signe inversé). On les compte d'abord, puis on construit une vue `clients_clean` qui exclut les doublons et applique `ABS(age)`.

In [ ]:
%%sql
SELECT
    COUNT(*) FILTER (WHERE id_client LIKE '%_DUP') AS doublons_dup,
    COUNT(*) FILTER (WHERE age < 0)                AS ages_negatifs,
    COUNT(*)                                       AS total_brut
FROM clients_raw;

In [ ]:
%%sql
CREATE OR REPLACE VIEW clients AS
SELECT
    id_client,
    nom,
    prenom,
    sexe,
    ABS(age)                                AS age,
    ville,
    code_offre,
    CAST(date_souscription AS DATE)         AS date_souscription,
    CAST(date_resiliation  AS DATE)         AS date_resiliation,
    statut
FROM clients_raw
WHERE id_client NOT LIKE '%_DUP';

SELECT COUNT(*) AS clients_propres FROM clients;

### 1.2 Montants nuls dans `factures`

#### 🔧 MÉTHODE

2 % des factures ont un montant à 0 — soit erreur de saisie, soit promo gratuite. On les exclut du calcul d'ARPU mais on les conserve pour la volumétrie. Vue `factures` typée pour la suite.

In [ ]:
%%sql
SELECT
    COUNT(*)                                       AS total,
    COUNT(*) FILTER (WHERE montant_fcfa = 0)       AS montants_nuls,
    ROUND(100.0 * COUNT(*) FILTER (WHERE montant_fcfa = 0) / COUNT(*), 2) AS pct_nuls
FROM factures_raw;

In [ ]:
%%sql
CREATE OR REPLACE VIEW factures AS
SELECT
    id_facture,
    id_client,
    CAST(date_facturation AS DATE)         AS date_facturation,
    CAST(montant_fcfa AS INTEGER)          AS montant_fcfa,
    methode_paiement
FROM factures_raw;

### 1.3 Délais de résolution négatifs (`reclamations`)

#### 🔧 MÉTHODE

Un délai négatif (résolu *avant* d'être ouvert) est physiquement impossible — c'est un bug de saisie côté centre d'appels. On le corrige par `ABS()`. La vue `reclamations` typée date est créée.

In [ ]:
%%sql
SELECT
    COUNT(*)                                                 AS total,
    COUNT(*) FILTER (WHERE delai_resolution_jours < 0)       AS delais_negatifs,
    COUNT(*) FILTER (WHERE delai_resolution_jours IS NULL)   AS delais_null
FROM reclamations_raw;

In [ ]:
%%sql
CREATE OR REPLACE VIEW reclamations AS
SELECT
    id_ticket,
    id_client,
    CAST(date_creation AS DATE)            AS date_creation,
    type,
    statut,
    CASE WHEN delai_resolution_jours < 0
         THEN ABS(delai_resolution_jours)
         ELSE delai_resolution_jours END   AS delai_resolution_jours
FROM reclamations_raw;

> 💡 **INTERPRÉTATION** *(à finaliser)*
>
> - Doublons : 30 supprimés sur ~8 030 (0,4 %).
> - Âges négatifs : 5 corrigés via `ABS()`.
> - Montants nuls factures : ~2 770 lignes (2 % de 138 284).
> - Délais négatifs : ~290 corrigés (3 % des ~9 791 réclamations).

> 🏥 **MÉTIER** — Ces taux d'anomalies sont compatibles avec un opérateur réel (les systèmes de billing facturent parfois en double, les agents call-center mal-saisissent). Sans ce nettoyage, l'ARPU global serait sous-estimé de ~2 % et le délai moyen de résolution serait absurde (médiane potentiellement négative).

---
## 2. KPIs globaux du churn

On commence par les chiffres qui apparaissent en page 1 du dashboard exécutif : taux de churn cumulatif sur la période, ARPU global, anciennetés comparées.

### 2.1 Taux de churn cumulatif et répartition statut

#### 🔧 MÉTHODE

Le **taux de churn cumulatif** = `nb_resilies / nb_total`. C'est la part de clients qui ont quitté IvoirCom à un moment quelconque sur les 24 mois observés. À ne pas confondre avec le **taux de churn trimestriel** (12 % annoncé par la DG) qui se calcule sur une fenêtre glissante de 90 jours.

In [ ]:
%%sql
SELECT
    statut,
    COUNT(*)                                                  AS n_clients,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)        AS pct
FROM clients
GROUP BY statut
ORDER BY n_clients DESC;

### 2.2 ARPU global, médiane et écart actifs vs churners

#### 🔧 MÉTHODE

L'**ARPU** (Average Revenue Per User) se calcule sur les factures non nulles. La médiane est plus robuste que la moyenne pour une distribution asymétrique (queue longue côté offres Pro). L'écart actifs vs churners est un *signal de valeur perdue* : si les churners avaient un ARPU plus faible, c'est une perte « facile » à compenser ; s'ils avaient un ARPU élevé, c'est une perte stratégique.

In [ ]:
%%sql
WITH arpu_par_client AS (
    SELECT
        f.id_client,
        AVG(f.montant_fcfa) AS arpu_mensuel
    FROM factures f
    WHERE f.montant_fcfa > 0
    GROUP BY f.id_client
)
SELECT
    c.statut,
    COUNT(*)                                       AS n_clients,
    ROUND(AVG(a.arpu_mensuel), 0)                  AS arpu_moyen_fcfa,
    ROUND(MEDIAN(a.arpu_mensuel), 0)               AS arpu_median_fcfa,
    ROUND(MIN(a.arpu_mensuel), 0)                  AS arpu_min,
    ROUND(MAX(a.arpu_mensuel), 0)                  AS arpu_max
FROM clients c
JOIN arpu_par_client a USING (id_client)
GROUP BY c.statut
ORDER BY arpu_moyen_fcfa DESC;

### 2.3 Ancienneté moyenne actifs vs churners

#### 🔧 MÉTHODE

L'ancienneté = `date_resiliation - date_souscription` pour les churners, `today - date_souscription` pour les actifs. On utilise `'2025-12-31'` comme date pivot pour les actifs afin de figer la comparaison. Si les churners partent jeunes (< 6 mois), c'est un échec d'onboarding ; s'ils partent matures (> 18 mois), c'est un signal de saturation/concurrence.

In [ ]:
%%sql
SELECT
    statut,
    COUNT(*) AS n_clients,
    ROUND(AVG(
        DATE_DIFF('day',
                  date_souscription,
                  COALESCE(date_resiliation, DATE '2025-12-31'))
        / 30.0
    ), 1) AS anciennete_moyenne_mois,
    ROUND(MEDIAN(
        DATE_DIFF('day',
                  date_souscription,
                  COALESCE(date_resiliation, DATE '2025-12-31'))
        / 30.0
    ), 1) AS anciennete_mediane_mois
FROM clients
GROUP BY statut;

> 💡 **INTERPRÉTATION** *(à finaliser après exécution)*
>
> - Taux churn cumulatif : ~XX % (placeholder — ~25 % attendu).
> - ARPU global moyen : ~X XXX FCFA, médian ~X XXX FCFA.
> - Écart ARPU actifs vs churners : ~+X % chez les actifs.
> - Ancienneté moyenne churners : ~XX mois vs ~XX mois actifs.

> 🏥 **MÉTIER** — Si les churners ont un ARPU significativement inférieur, IvoirCom perd surtout de la **base bas de gamme** (offres Pulse, Étudiant). C'est moins critique financièrement mais ça érode la part de marché. Si l'écart est faible, le churn touche aussi les offres premium — il faut alors prioritairement la rétention.

---
## 3. Segmentation : qui churne le plus ?

On éclate le taux de churn cumulatif par dimensions opérationnelles : **offre**, **ville**, **tranche d'âge**, **sexe**. L'objectif est d'identifier 2-3 segments hautement actionnables.

### 3.1 Taux de churn par offre

#### 🔧 MÉTHODE

Une jointure `clients ⋈ offres` permet d'enrichir avec le libellé et le prix. Le taux de churn par offre = `nb_resilies / nb_total` *au sein* de chaque offre. On trie descendant pour faire émerger les offres « porte-tournante ».

In [ ]:
%%sql
SELECT
    o.code_offre,
    o.libelle,
    o.prix_fcfa,
    COUNT(*)                                                AS n_total,
    COUNT(*) FILTER (WHERE c.statut = 'resilie')            AS n_churners,
    ROUND(100.0 * COUNT(*) FILTER (WHERE c.statut = 'resilie') / COUNT(*), 1) AS taux_churn_pct
FROM clients c
JOIN offres o USING (code_offre)
GROUP BY o.code_offre, o.libelle, o.prix_fcfa
ORDER BY taux_churn_pct DESC;

In [ ]:
df_offre = %sql SELECT o.code_offre, o.libelle, COUNT(*) AS n_total, COUNT(*) FILTER (WHERE c.statut = 'resilie') AS n_churners, ROUND(100.0 * COUNT(*) FILTER (WHERE c.statut = 'resilie') / COUNT(*), 1) AS taux_churn_pct FROM clients c JOIN offres o USING (code_offre) GROUP BY o.code_offre, o.libelle ORDER BY taux_churn_pct DESC;

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(df_offre["libelle"], df_offre["taux_churn_pct"], color=COLORS["primary"])
ax.axvline(25, color=COLORS["danger"], linestyle="--", label="Seuil critique 25 %")
ax.set_xlabel("Taux de churn cumulatif (%)")
ax.set_title("Taux de churn par offre — IvoirCom 2024-2025")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 3.2 Taux de churn par ville et par tranche d'âge

#### 🔧 MÉTHODE

On découpe l'âge en tranches métier (18-25, 26-35, 36-45, 46-60, 60+). En SQL, on utilise `CASE WHEN`. Le `GROUP BY ROLLUP` pourrait calculer aussi le total — mais ici on garde une simple agrégation et on ajoutera la totale en pandas pour la lisibilité.

In [ ]:
%%sql
SELECT
    ville,
    COUNT(*)                                              AS n_total,
    COUNT(*) FILTER (WHERE statut = 'resilie')            AS n_churners,
    ROUND(100.0 * COUNT(*) FILTER (WHERE statut = 'resilie') / COUNT(*), 1) AS taux_churn_pct
FROM clients
GROUP BY ville
ORDER BY taux_churn_pct DESC;

In [ ]:
%%sql
SELECT
    CASE
        WHEN age BETWEEN 18 AND 25 THEN '1. 18-25'
        WHEN age BETWEEN 26 AND 35 THEN '2. 26-35'
        WHEN age BETWEEN 36 AND 45 THEN '3. 36-45'
        WHEN age BETWEEN 46 AND 60 THEN '4. 46-60'
        ELSE                              '5. 60+'
    END                                                   AS tranche_age,
    COUNT(*)                                              AS n_total,
    COUNT(*) FILTER (WHERE statut = 'resilie')            AS n_churners,
    ROUND(100.0 * COUNT(*) FILTER (WHERE statut = 'resilie') / COUNT(*), 1) AS taux_churn_pct
FROM clients
GROUP BY tranche_age
ORDER BY tranche_age;

### 3.3 Heatmap croisée Offre × Ville (taux de churn)

#### 🔧 MÉTHODE

Une heatmap est l'outil idéal pour repérer une **interaction** : par exemple, l'offre Pulse churn-t-elle particulièrement à Bouaké ? Le croisement révèle des poches d'attrition non visibles dans les marges. On utilise `seaborn.heatmap` avec une palette divergente centrée sur le taux moyen.

In [ ]:
df_cross = %sql SELECT ville, code_offre, ROUND(100.0 * COUNT(*) FILTER (WHERE statut = 'resilie') / COUNT(*), 1) AS taux_churn FROM clients GROUP BY ville, code_offre;

pivot = df_cross.pivot(index="ville", columns="code_offre", values="taux_churn")

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
            center=25, cbar_kws={"label": "Taux churn (%)"}, ax=ax)
ax.set_title("Heatmap taux de churn — Ville x Offre")
plt.tight_layout()
plt.show()

> 💡 **INTERPRÉTATION** *(à finaliser)*
>
> - Top 2 offres à risque : ___ et ___ (~XX % churn chacune).
> - Top ville à risque : ___ (~XX %).
> - Tranche d'âge la plus churnante : ___.
> - Cellule la plus rouge dans la heatmap : ___ × ___.

> 🏥 **MÉTIER** — Une offre d'entrée de gamme avec un churn > 30 % est typiquement un signal qu'elle ne **convertit pas vers le haut** (les utilisateurs essaient puis quittent au lieu de migrer vers une offre supérieure). Une ville secondaire avec un churn élevé peut signaler un déficit de couverture réseau.

---
## 4. Analyse de cohortes — quelle génération de souscripteurs tient le mieux ?

Une **cohorte** = ensemble des clients souscrits le même mois. La courbe de rétention par cohorte révèle si la dégradation du churn est **structurelle** (toutes les cohortes pareilles) ou **cyclique** (certaines cohortes chutent vite à cause d'un événement).

### 4.1 Construction de la table de cohortes

#### 🔧 MÉTHODE

Pour chaque (cohorte, période d'observation), on calcule la part des clients de la cohorte encore actifs à cette période. Logique :

1. `cohorte_mois` = `DATE_TRUNC('month', date_souscription)`
2. Pour chaque mois M depuis la souscription, on flag *actif* si `date_resiliation` n'est pas encore atteinte ou est NULL.
3. On agrège par `(cohorte_mois, mois_depuis_souscription)`.

On limite aux cohortes 2024 (12 cohortes) pour avoir au moins 12 mois d'observation.

In [ ]:
%%sql
CREATE OR REPLACE VIEW cohortes AS
WITH base AS (
    SELECT
        id_client,
        DATE_TRUNC('month', date_souscription)         AS cohorte_mois,
        date_souscription,
        COALESCE(date_resiliation, DATE '2025-12-31')  AS date_fin
    FROM clients
    WHERE EXTRACT(YEAR FROM date_souscription) = 2024
),
expand AS (
    SELECT
        b.id_client,
        b.cohorte_mois,
        DATE_TRUNC('month', g.gen_date)                AS mois_observation,
        DATE_DIFF('month', b.cohorte_mois, g.gen_date) AS mois_depuis_souscription
    FROM base b,
         generate_series(b.date_souscription, b.date_fin, INTERVAL 1 MONTH) AS g(gen_date)
)
SELECT
    cohorte_mois,
    mois_depuis_souscription,
    COUNT(DISTINCT id_client)                          AS clients_actifs
FROM expand
WHERE mois_depuis_souscription BETWEEN 0 AND 11
GROUP BY cohorte_mois, mois_depuis_souscription
ORDER BY cohorte_mois, mois_depuis_souscription;

SELECT * FROM cohortes LIMIT 24;

### 4.2 Pivot et heatmap de rétention

#### 🔧 MÉTHODE

On pivote en table 12×12 : lignes = cohorte, colonnes = mois écoulé. Chaque cellule = % de clients de la cohorte encore actifs. Le mois 0 est toujours 100 %, puis la courbe descend.

In [ ]:
df_coh = %sql SELECT * FROM cohortes;

# normaliser en % par cohorte (mois 0 = 100 %)
df_coh["cohorte_mois"] = pd.to_datetime(df_coh["cohorte_mois"])
df_coh["cohorte_label"] = df_coh["cohorte_mois"].dt.strftime("%Y-%m")

size_at_0 = df_coh[df_coh["mois_depuis_souscription"] == 0]\
    .set_index("cohorte_label")["clients_actifs"]
df_coh["taux_retention_pct"] = df_coh.apply(
    lambda r: 100 * r["clients_actifs"] / size_at_0[r["cohorte_label"]],
    axis=1
)

pivot = df_coh.pivot(index="cohorte_label",
                     columns="mois_depuis_souscription",
                     values="taux_retention_pct")

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="RdYlGn",
            vmin=50, vmax=100,
            cbar_kws={"label": "Taux de retention (%)"}, ax=ax)
ax.set_xlabel("Mois depuis souscription")
ax.set_ylabel("Cohorte")
ax.set_title("Heatmap de retention par cohorte — souscriptions 2024")
plt.tight_layout()
plt.show()

> 💡 **INTERPRÉTATION** *(à finaliser après lecture de la heatmap)*
>
> - Rétention médiane à M+6 : ~XX %.
> - Rétention médiane à M+12 : ~XX %.
> - Cohorte la plus solide : ~2024-XX (rétention M+12 ~XX %).
> - Cohorte la plus fragile : ~2024-XX (rétention M+12 ~XX %).

> 🏥 **MÉTIER** — Si la rétention M+3 est inférieure à 90 %, le problème est dans l'**onboarding** (premier mois mal géré, attentes non tenues). Si la dégradation est régulière de M+3 à M+12, c'est un problème **structurel** (offre commoditisée, prix, concurrence).

---
## 5. Signaux faibles — les réclamations prédisent-elles le départ ?

Hypothèse métier : un client qui ouvre un ticket et ne reçoit pas de résolution (ticket *abandonné* ou *ouvert > 30j*) part dans les semaines qui suivent. On va le vérifier en SQL.

### 5.1 Top types de réclamations et statut de résolution

#### 🔧 MÉTHODE

Croisement `type × statut` avec `COUNT(*)` et `% lignes` via window function `SUM() OVER ()`. Ça donne immédiatement le mix volume × qualité de traitement.

In [ ]:
%%sql
SELECT
    type,
    statut,
    COUNT(*)                                                       AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY type), 1) AS pct_dans_type
FROM reclamations
GROUP BY type, statut
ORDER BY type, n DESC;

### 5.2 Comparaison nb tickets churners vs actifs

#### 🔧 MÉTHODE

On compte le nombre de tickets par client, on joint avec `clients` pour récupérer le statut, puis on calcule la moyenne par groupe. Si les churners ont en moyenne 2x plus de tickets que les actifs, c'est confirmé que la frustration mène au départ.

In [ ]:
%%sql
WITH tickets_par_client AS (
    SELECT
        id_client,
        COUNT(*)                                              AS n_tickets,
        COUNT(*) FILTER (WHERE statut = 'abandonne')          AS n_abandonnes,
        COUNT(*) FILTER (WHERE statut = 'ouvert')             AS n_ouverts
    FROM reclamations
    GROUP BY id_client
)
SELECT
    c.statut                                                  AS statut_client,
    COUNT(*)                                                  AS n_clients,
    ROUND(AVG(COALESCE(t.n_tickets, 0)), 2)                   AS tickets_moyens,
    ROUND(AVG(COALESCE(t.n_abandonnes, 0)), 2)                AS abandonnes_moyens,
    ROUND(AVG(COALESCE(t.n_ouverts, 0)), 2)                   AS ouverts_moyens,
    ROUND(100.0 * COUNT(*) FILTER (WHERE t.n_tickets >= 2) / COUNT(*), 1) AS pct_avec_2plus_tickets
FROM clients c
LEFT JOIN tickets_par_client t USING (id_client)
GROUP BY c.statut
ORDER BY tickets_moyens DESC;

### 5.3 Test du signal : taux de churn parmi clients à 2+ tickets non résolus

#### 🔧 MÉTHODE

On isole les clients qui ont au moins 2 tickets en statut *ouvert* ou *abandonné* (= signal d'insatisfaction prolongée). On compare leur taux de churn à la base globale. Si l'écart est > 2x, le signal est fort et opérationnel : *« contacter les clients à 2+ tickets non résolus avant qu'ils ne partent »*.

In [ ]:
%%sql
WITH clients_a_risque AS (
    SELECT id_client
    FROM reclamations
    WHERE statut IN ('ouvert', 'abandonne')
    GROUP BY id_client
    HAVING COUNT(*) >= 2
)
SELECT
    'Clients a 2+ tickets non resolus'                AS segment,
    COUNT(*)                                          AS n,
    COUNT(*) FILTER (WHERE c.statut = 'resilie')      AS n_churners,
    ROUND(100.0 * COUNT(*) FILTER (WHERE c.statut = 'resilie') / COUNT(*), 1) AS taux_churn_pct
FROM clients c
JOIN clients_a_risque USING (id_client)
UNION ALL
SELECT
    'Base globale',
    COUNT(*),
    COUNT(*) FILTER (WHERE statut = 'resilie'),
    ROUND(100.0 * COUNT(*) FILTER (WHERE statut = 'resilie') / COUNT(*), 1)
FROM clients;

> 💡 **INTERPRÉTATION** *(à finaliser)*
>
> - Top type de réclamation : ___ (~XX % du volume).
> - Tickets moyens churners vs actifs : ~XX vs XX (ratio xX).
> - Taux churn parmi 2+ tickets non résolus : ~XX % vs XX % base globale (xX écart).

> 🏥 **MÉTIER** — Le signal *« 2+ tickets non résolus = clients à appeler aujourd'hui »* est le levier de rétention le plus rapide à mettre en place : il ne nécessite pas de modèle ML, juste un script SQL hebdomadaire et 5-10 conseillers dédiés.

---
## 6. Segmentation RFM Telecom — qui sont mes Champions, mes At-Risk, mes Lost ?

Le RFM télécom adapte la grille e-commerce :

- **R** (Récence) : jours depuis la dernière facture ; petit = récent = bon.
- **F** (Fréquence) : *inversée* — nb de réclamations sur 6 mois ; petit = peu de plaintes = bon.
- **M** (Monétaire) : ARPU des 6 derniers mois ; grand = bon.

On `NTILE(5)` chaque dimension (1 = pire, 5 = meilleur) puis on combine en segments lisibles.

### 6.1 Calcul des 3 dimensions RFM

#### 🔧 MÉTHODE

Date pivot = `2025-12-31`. Fenêtre 6 derniers mois = `[2025-07-01, 2025-12-31]`. On joint 3 sous-requêtes (recence, frequence, monetaire) à `clients`, on applique `NTILE(5)` partition par dimension.

In [ ]:
%%sql
CREATE OR REPLACE TABLE rfm AS
WITH date_pivot AS (
    SELECT DATE '2025-12-31' AS d_pivot
),
recence AS (
    SELECT
        f.id_client,
        DATE_DIFF('day', MAX(f.date_facturation), p.d_pivot) AS jours_depuis_derniere_facture
    FROM factures f, date_pivot p
    GROUP BY f.id_client, p.d_pivot
),
frequence AS (
    SELECT
        r.id_client,
        COUNT(*) AS n_reclamations_6m
    FROM reclamations r
    WHERE r.date_creation >= DATE '2025-07-01'
    GROUP BY r.id_client
),
monetaire AS (
    SELECT
        f.id_client,
        AVG(f.montant_fcfa) AS arpu_6m
    FROM factures f
    WHERE f.date_facturation >= DATE '2025-07-01'
      AND f.montant_fcfa > 0
    GROUP BY f.id_client
)
SELECT
    c.id_client,
    c.code_offre,
    c.ville,
    c.statut,
    COALESCE(r.jours_depuis_derniere_facture, 999) AS jours_depuis_derniere_facture,
    COALESCE(f.n_reclamations_6m, 0)               AS n_reclamations_6m,
    COALESCE(m.arpu_6m, 0)                         AS arpu_6m,
    NTILE(5) OVER (ORDER BY COALESCE(r.jours_depuis_derniere_facture, 999) DESC) AS R_score,
    NTILE(5) OVER (ORDER BY COALESCE(f.n_reclamations_6m, 0) DESC)               AS F_score,
    NTILE(5) OVER (ORDER BY COALESCE(m.arpu_6m, 0))                              AS M_score
FROM clients c
LEFT JOIN recence    r USING (id_client)
LEFT JOIN frequence  f USING (id_client)
LEFT JOIN monetaire  m USING (id_client);

SELECT * FROM rfm LIMIT 10;

### 6.2 Attribution de segments RFM

#### 🔧 MÉTHODE

Règles métier classiques :

- **Champions** : R≥4, F≥4, M≥4 (récents, peu de plaintes, ARPU élevé)
- **Loyal** : R≥3, F≥3, M≥3
- **At Risk** : R≤2 et M≥3 (n'ont pas facturé récemment mais étaient bons clients)
- **Lost** : R≤2 et M≤2
- **New** : R≥4, M≤2 (récent mais pas encore prouvé)
- **Others** : tous les autres

In [ ]:
%%sql
WITH segments AS (
    SELECT *,
        CASE
            WHEN R_score >= 4 AND F_score >= 4 AND M_score >= 4 THEN 'Champions'
            WHEN R_score >= 3 AND F_score >= 3 AND M_score >= 3 THEN 'Loyal'
            WHEN R_score <= 2 AND M_score >= 3                  THEN 'At Risk'
            WHEN R_score <= 2 AND M_score <= 2                  THEN 'Lost'
            WHEN R_score >= 4 AND M_score <= 2                  THEN 'New'
            ELSE                                                     'Others'
        END AS segment_rfm
    FROM rfm
)
SELECT
    segment_rfm,
    COUNT(*)                                                  AS n_clients,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)        AS pct_base,
    ROUND(AVG(arpu_6m), 0)                                    AS arpu_6m_moyen,
    ROUND(AVG(jours_depuis_derniere_facture), 0)              AS recence_moyenne_j,
    ROUND(AVG(n_reclamations_6m), 2)                          AS plaintes_6m,
    ROUND(100.0 * COUNT(*) FILTER (WHERE statut = 'resilie') / COUNT(*), 1) AS taux_churn_pct
FROM segments
GROUP BY segment_rfm
ORDER BY n_clients DESC;

### 6.3 Top 50 clients At Risk les plus prioritaires

#### 🔧 MÉTHODE

Au sein du segment *At Risk*, on classe par ARPU 6 mois descendant. Les premiers de la liste représentent **la valeur perdue prioritaire** : ce sont les clients qu'il faut appeler aujourd'hui.

In [ ]:
%%sql
WITH segments AS (
    SELECT *,
        CASE
            WHEN R_score <= 2 AND M_score >= 3 THEN 'At Risk'
            ELSE                                    'Other'
        END AS segment_rfm
    FROM rfm
)
SELECT
    id_client,
    code_offre,
    ville,
    statut,
    jours_depuis_derniere_facture,
    n_reclamations_6m,
    ROUND(arpu_6m, 0) AS arpu_6m,
    R_score,
    F_score,
    M_score
FROM segments
WHERE segment_rfm = 'At Risk'
ORDER BY arpu_6m DESC
LIMIT 50;

> 💡 **INTERPRÉTATION** *(à finaliser)*
>
> - Champions : ~XX % de la base, ARPU moyen ~XXX FCFA, churn ~X %.
> - At Risk : ~XX % de la base, ARPU moyen ~XXX FCFA, churn ~XX %.
> - Lost : ~XX %.
> - Top At Risk : ~50 clients à fort ARPU à recontacter en priorité.

> 🏥 **MÉTIER** — Une bonne base devrait avoir 15-25 % de Champions et < 10 % d'At Risk. Si l'At Risk dépasse 15 %, on a une attrition de valeur élevée à venir.

---
## 7. Synthèse exécutive et 3 recommandations

### Tableau de bord récapitulatif (à finaliser après exécution)

| Indicateur | Valeur | Cible interne | Statut |
|---|---|---|---|
| Taux de churn cumulatif | XX % | < 20 % | 🔴 / 🟠 / 🟢 |
| ARPU global mensuel | X XXX FCFA | > 8 000 FCFA | 🔴 / 🟠 / 🟢 |
| Écart ARPU actifs vs churners | +XX % | — | indicatif |
| Taux churn offre la plus à risque | XX % | < 25 % | 🔴 / 🟠 / 🟢 |
| Rétention M+12 cohorte 2024 (médiane) | XX % | > 80 % | 🔴 / 🟠 / 🟢 |
| Taux churn parmi 2+ tickets non résolus | XX % | < 30 % | 🔴 / 🟠 / 🟢 |
| Part At Risk dans la base | XX % | < 10 % | 🔴 / 🟠 / 🟢 |

### Recommandations pour la direction commerciale d'IvoirCom

1. **Recontacter les ~XX clients à 2+ tickets non résolus** — ROI immédiat, pas de modèle ML requis. Estimation : X% de cette population évite le churn = ~Y M FCFA récupérés sur 6 mois.
2. **Plan de migration offre Pulse → Connect** — la plus churnante (~XX %). Offrir 1 mois gratuit Connect aux Pulse de >6 mois d'ancienneté qui consomment > 700 Mo. Estimation : -X pp de churn sur la cohorte.
3. **Programme At Risk** — ~50 clients à fort ARPU identifiés en RFM, appel commercial sortant + offre fidélité 3 mois. Estimation : ~Z M FCFA d'ARPU défendu.

*(les chiffres précis seront calés à la finalisation, après ton retour sur les outputs)*

---
<div style="background:#1E3A5F;padding:24px 32px;border-radius:10px;color:#FFFFFF;font-family:Georgia,serif;text-align:center;">
<div style="font-size:22px;font-weight:700;margin-bottom:6px;">Customer Churn Analytics — IvoirCom</div>
<div style="font-size:13px;color:#CBD5E0;font-family:'Segoe UI',sans-serif;"><b>DataProjectLab</b> — apprendre la data sur des cas concrets, structurés et orientés métier.</div>
</div>